# Ablation Study Demo Notebook
## Augmented LLM Inference for Bioinformatics Code Generation

> **Purpose:** This notebook gives a compact end-to-end demo of your project using the
> same pipeline structure as the full experiment, but on 4 real benchmark problems and
> with a deterministic mock model so it runs quickly during a presentation.

---

### Demo Scope
This notebook is structured around your **three Ollama models**:
- `ollama:qwen2.5:0.5b`
- `ollama:qwen2.5-coder:1.5b`
- `ollama:phi3:mini`

The notebook itself does **not** run those live models by default. Instead, it demonstrates:
1. how generated code is tested,
2. how the 5 augmentation conditions change the prompt,
3. how the experiment loop works,
4. and how results are summarised.

### Notebook Structure
| Section | What you will see |
|---------|-------------------|
| **1. Setup** | Import modules, confirm the environment works |
| **2. Code Executor** | Passing, failing, and timeout behaviour in the sandbox |
| **3. Augmentation Pipeline** | All 5 prompt conditions rendered for a real problem |
| **4. Model Interface** | How your 3 Ollama model names route through the interface layer |
| **5. Full Mini Experiment** | A 4-problem demo across all 5 conditions |
| **6. Results Visualisation** | Pass@1, gain over baseline, token efficiency, repair convergence |


---
## 1. Setup

Run this cell first. It imports the project modules and sets up logging.
Make sure `experiment_runner_patched.py`, `model_interface.py`, `augmentation_pipeline_patched.py`,
and `code_executor.py` are in the same directory as this notebook.


In [ ]:
import sys, os, json, textwrap, subprocess, logging, tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML, Markdown

# Set up clean logging output for the notebook
logging.basicConfig(level=logging.WARNING)  # suppress INFO spam in demos

# Import project modules
from code_executor import CodeExecutor
from augmentation_pipeline_patched import AugmentationPipeline
from experiment_runner_patched import extract_code
# model_interface and experiment_runner imported in later sections

print('All imports successful')
print(f'Python {sys.version}')


### ✅ Windows / Cross-platform Note

The `code_executor.py` file previously hardcoded `/tmp/` as the temp directory, which does not exist on Windows.
This has been fixed — it now uses `tempfile.gettempdir()` which resolves to the correct OS temp folder automatically:
- **Windows:** `C:\Users\<you>\AppData\Local\Temp\`
- **macOS / Linux:** `/tmp/`

The cell below verifies this is working correctly on your system.

In [ ]:
import tempfile, os
print(f'Temp directory on this system: {tempfile.gettempdir()}')

# Quick smoke-test: verify CodeExecutor works on this OS
executor = CodeExecutor(timeout=5)
test_code = 'def add(a, b): return a + b'
test_cases = [{'call': 'add(2, 3)', 'expected': 5}]
passed = executor.run_tests(test_code, test_cases)
print(f'CodeExecutor smoke test: {"✅ PASSED" if passed else "❌ FAILED"}') 

---
## 2. Code Executor ? The Sandbox Test Runner

`code_executor.py` is responsible for taking Python code generated by an LLM and
running it against test cases in an isolated subprocess.

This compact demo keeps only the three behaviours you most need to present:
- a correct solution that passes,
- a buggy solution that fails and produces repair feedback,
- a broken solution that gets killed by the timeout guard.


In [ ]:
executor = CodeExecutor(timeout=5)
print('CodeExecutor created with 5s default timeout')

### 2.1 — A Correct Implementation (should PASS ✅)

In [ ]:
correct_gc = '''
def gc_content(sequence: str) -> float:
    """Return the fraction of G and C bases in a DNA sequence."""
    sequence = sequence.upper()
    if not sequence:
        return 0.0
    gc = sum(1 for base in sequence if base in ('G', 'C'))
    return gc / len(sequence)
'''

tests = [
    {"call": "gc_content('ACGT')",   "expected": 0.5},
    {"call": "gc_content('GGCC')",   "expected": 1.0},
    {"call": "gc_content('AAAA')",   "expected": 0.0},
    {"call": "gc_content('')",       "expected": 0.0},
]

passed, error = executor.run_tests(correct_gc, tests, return_error=True)
status = '✅ PASSED' if passed else '❌ FAILED'
print(f'{status}')
if error:
    print(f'Error: {error}')

### 2.2 — A Buggy Implementation (should FAIL ❌)

In [ ]:
# Bug: only counts G, forgets C
buggy_gc = '''
def gc_content(sequence: str) -> float:
    sequence = sequence.upper()
    return sequence.count('G') / len(sequence) if sequence else 0.0
'''

passed, error = executor.run_tests(buggy_gc, tests, return_error=True)
status = '✅ PASSED' if passed else '❌ FAILED'
print(f'{status}')
print(f'Error output that would be sent to repair prompt:')
print('-' * 50)
print(error)

### 2.4 — Timeout Enforcement

In [ ]:
infinite_loop_code = '''
def gc_content(sequence: str) -> float:
    while True:  # infinite loop bug
        pass
    return 0.0
'''

short_executor = CodeExecutor(timeout=2)
import time
start = time.time()
passed, error = short_executor.run_tests(infinite_loop_code, tests, return_error=True)
elapsed = time.time() - start

print(f'Completed in {elapsed:.1f}s (killed by timeout)')
print(f'Passed: {passed}')
print(f'Error: {error}')

---
## 3. Augmentation Pipeline ? The 5 Prompt Conditions

This module constructs the prompt sent to the model. Each condition changes how much
context or structure is given before generation. Here we render all five prompts for the
same real problem so you can compare them directly.


In [ ]:
from pathlib import Path

# Load one real example per dataset category for the notebook demo
DATASET_PATH = Path('biocoder_dataset/python_problems.json')
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Missing dataset file: {DATASET_PATH}')

with open(DATASET_PATH, encoding='utf-8') as f:
    ALL_DEMO_PROBLEMS = json.load(f)

preferred_ids = ['bc_001', 'bc_021', 'bc_041', 'bc_061']
selected = []
seen_categories = set()
for target_id in preferred_ids:
    match = next((p for p in ALL_DEMO_PROBLEMS if p.get('id') == target_id), None)
    if match and match.get('category') not in seen_categories:
        selected.append(match)
        seen_categories.add(match.get('category'))

for problem in ALL_DEMO_PROBLEMS:
    category = problem.get('category')
    if category not in seen_categories:
        selected.append(problem)
        seen_categories.add(category)

DEMO_POOL = selected
DEMO_PROBLEM = DEMO_POOL[0]

def build_demo_prompt(condition: str, problem: dict, pool: list[dict]):
    """Generate notebook-safe prompts.

    For C4, keep the prompt structure the same but use three deterministic
    in-memory examples from the demo pool instead of loading an embedding model.
    This keeps the notebook runnable without external downloads.
    """
    pipeline = AugmentationPipeline(condition)
    if condition == 'C4_rag_context':
        local_examples = [p for p in pool if p.get('id') != problem.get('id')][:3]
        if not local_examples:
            local_examples = [problem]
        pipeline._retrieve_similar = lambda current_problem, all_problems, top_k=3: local_examples[:top_k]
    return pipeline.generate_prompt(problem=problem, model='demo', all_problems=pool)

print(f'Loaded {len(DEMO_POOL)} category demo problems from {DATASET_PATH}')
for problem in DEMO_POOL:
    print(f"  - {problem['category']:<18} {problem['id']}  {problem['signature']}")
print()
print('Representative prompt example:', DEMO_PROBLEM['signature'])
print('Notebook note: C4 uses deterministic local retrieval so the demo runs without external downloads.')


In [ ]:
CONDITIONS = [
    ('C1_zero_shot',        '?? C1 ? Zero-Shot Baseline'),
    ('C2_few_shot',         '?? C2 ? Few-Shot Examples'),
    ('C3_chain_of_thought', '?? C3 ? Chain-of-Thought'),
    ('C4_rag_context',      '?? C4 ? RAG Context'),
    ('C5_iterative_repair', '?? C5 ? Iterative Repair (initial prompt)'),
]

for cond_key, label in CONDITIONS:
    result = build_demo_prompt(cond_key, DEMO_PROBLEM, DEMO_POOL)
    prompt = result['prompt']
    print('=' * 70)
    print(f'{label}')
    print(f'Prompt length: {len(prompt)} chars | ~{len(prompt.split())} words')
    print('-' * 70)
    print(prompt[:800] + ('...[truncated]' if len(prompt) > 800 else ''))
    print()

### 3.1 — Prompt Length Comparison Across Conditions

In [ ]:
import matplotlib.pyplot as plt

cond_labels = ['C1\nZero-Shot', 'C2\nFew-Shot', 'C3\nCoT', 'C4\nRAG', 'C5\nRepair']
lengths = []
for cond_key, _ in CONDITIONS:
    r = build_demo_prompt(cond_key, DEMO_PROBLEM, DEMO_POOL)
    lengths.append(len(r['prompt'].split()))

colors = ['#3B82F6','#EAB308','#F97316','#8B5CF6','#EF4444']
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(cond_labels, lengths, color=colors, width=0.55, edgecolor='white')
ax.set_ylabel('Approximate Word Count', fontsize=11)
ax.set_title('Prompt Size by Condition (same problem)', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(lengths) * 1.25)
for bar, val in zip(bars, lengths):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{val}w', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
print('Note: RAG is largest due to API context block + 3 locally retrieved demo examples')

---
## 4. Model Interface ? Your 3-Model Ollama Setup

`model_interface.py` is the abstraction layer that turns a model name into the right
backend client. In your project, all three target models are local Ollama models, so they
all route through the same `OllamaInterface` class.


In [ ]:
from model_interface import ModelInterface

# Show exactly the 3 model identifiers used in your project
project_models = [
    'ollama:qwen2.5:0.5b',
    'ollama:qwen2.5-coder:1.5b',
    'ollama:phi3:mini',
]

print('Model identifier  ->  Interface class')
print('-' * 50)
for model_str in project_models:
    interface = 'OllamaInterface' if model_str.startswith('ollama:') else 'OtherInterface'
    print(f'  {model_str:<35} ->  {interface}')

print('\nPresentation note: the notebook demo below uses a mock model for speed and reliability,')
print('but the full experiment uses these exact model strings with the same runner structure.')


---
## 5. Full Mini Experiment - End-to-End Demo (Mocked)

This section demonstrates the full experiment loop on 4 problems,
one from each dataset category, without making live Ollama calls.
A mock model is used so the demo runs quickly and predictably during a presentation.


In [ ]:
# Mock model that simulates different behaviour per condition

MOCK_SOLUTIONS = {
    'gc_content': {
        'correct': '''def gc_content(sequence: str) -> float:
    sequence = sequence.upper()
    if not sequence:
        return 0.0
    return (sequence.count('G') + sequence.count('C')) / len(sequence)''',
        'buggy': '''def gc_content(sequence: str) -> float:
    if not sequence:
        return 0.0
    return sequence.count('G') / len(sequence)  # bug: forgot C'''
    },
    'translate_codon': {
        'correct': '''def translate_codon(codon: str) -> str:
    table = {'UUU':'F','UUC':'F','UUA':'L','UUG':'L','CUU':'L','CUC':'L','CUA':'L','CUG':'L','AUU':'I','AUC':'I','AUA':'I','AUG':'M','GUU':'V','GUC':'V','GUA':'V','GUG':'V','UCU':'S','UCC':'S','UCA':'S','UCG':'S','CCU':'P','CCC':'P','CCA':'P','CCG':'P','ACU':'T','ACC':'T','ACA':'T','ACG':'T','GCU':'A','GCC':'A','GCA':'A','GCG':'A','UAU':'Y','UAC':'Y','UAA':'*','UAG':'*','CAU':'H','CAC':'H','CAA':'Q','CAG':'Q','AAU':'N','AAC':'N','AAA':'K','AAG':'K','GAU':'D','GAC':'D','GAA':'E','GAG':'E','UGU':'C','UGC':'C','UGA':'*','UGG':'W','CGU':'R','CGC':'R','CGA':'R','CGG':'R','AGU':'S','AGC':'S','AGA':'R','AGG':'R','GGU':'G','GGC':'G','GGA':'G','GGG':'G'}
    return table.get(codon.upper(), 'X')''',
        'buggy': '''def translate_codon(codon: str) -> str:
    return 'X'  # bug: ignores the codon table'''
    },
    'parse_fasta_string': {
        'correct': '''def parse_fasta_string(fasta: str) -> dict:
    result = {}
    current_id = None
    for line in fasta.strip().splitlines():
        line = line.strip()
        if line.startswith('>'):
            current_id = line[1:].split()[0]
            result[current_id] = ''
        elif current_id:
            result[current_id] += line
    return result''',
        'buggy': '''def parse_fasta_string(fasta: str) -> dict:
    return {}  # bug: never parses any records'''
    },
    'mean': {
        'correct': '''def mean(values: list) -> float:
    if not values:
        return 0.0
    return sum(values) / len(values)''',
        'buggy': '''def mean(values: list) -> float:
    return sum(values)  # bug: forgot to divide by length'''
    },
}

class MockModelInterface:
    'Simulates condition-sensitive behavior with reasoning-heavy outputs.'
    def __init__(self, name='mock_model'):
        self.name = name
        self.call_count = 0

    def generate(self, prompt: str, max_tokens: int = 2048):
        self.call_count += 1
        prompt_lower = prompt.lower()

        import re

        fn_name = 'gc_content'
        fn_matches = re.findall(r"def\s+([A-Za-z_]\w*)\s*\(", prompt)
        if fn_matches:
            fn_name = fn_matches[-1]

        if '[c5 initial attempt]' in prompt_lower:
            code = MOCK_SOLUTIONS[fn_name]['buggy']
            text = f"First repair attempt reasoning.\n\n{code}\n\nThis version is still wrong."
        elif 'diagnose' in prompt_lower or 'fix this code' in prompt_lower:
            code = MOCK_SOLUTIONS[fn_name]['correct']
            text = f"Repair reasoning.\n\n{code}\n\nFixed."
        elif 'use the following context' in prompt_lower:
            code = MOCK_SOLUTIONS[fn_name]['correct']
            text = f"Retrieved context points to the right implementation.\n\n{code}\n\nDone."
        elif 'step by step' in prompt_lower:
            code = MOCK_SOLUTIONS[fn_name]['correct']
            text = f"Reasoning first.\n\n{code}\n\nDone."
        else:
            code = MOCK_SOLUTIONS[fn_name]['buggy']
            text = code

        return {
            'text': text,
            'prompt_tokens': len(prompt.split()),
            'completion_tokens': len(text.split()),
        }

print('MockModelInterface defined')


In [ ]:
# One demo problem per dataset category

DEMO_PROBLEMS = DEMO_POOL
print(f'{len(DEMO_PROBLEMS)} demo problems loaded')
for problem in DEMO_PROBLEMS:
    print(f"  - {problem['category']:<18} {problem['id']}  {problem['signature']}")


In [ ]:
# Run the mini experiment

CONDITIONS_DEMO = [
    'C1_zero_shot',
    'C2_few_shot',
    'C3_chain_of_thought',
    'C4_rag_context',
    'C5_iterative_repair',
]

executor = CodeExecutor(timeout=5)
mock_model = MockModelInterface()

results_table = []

print(f'Running mini experiment: {len(CONDITIONS_DEMO)} conditions x {len(DEMO_PROBLEMS)} problems\n')

for condition in CONDITIONS_DEMO:
    cond_passed = 0
    print(f'  Condition: {condition}')

    for problem in DEMO_PROBLEMS:
        prompt_data = build_demo_prompt(condition, problem, DEMO_PROBLEMS)
        model_prompt = prompt_data['prompt']
        if condition == 'C5_iterative_repair':
            model_prompt = '[C5 initial attempt]\n\n' + model_prompt

        response = mock_model.generate(model_prompt)
        generated_code = extract_code(response['text'])
        attempts = 1

        if condition == 'C5_iterative_repair':
            passed_first, error = executor.run_tests(generated_code, problem['tests'], return_error=True)
            if not passed_first:
                repair_prompt = (
                    f'The code failed:\n{error}\n\n'
                    f'Task: {problem["docstring"]}\n\n'
                    f'Fix this code:\n```python\n{generated_code}\n```\n\nDiagnose and fix.'
                )
                repair_response = mock_model.generate(repair_prompt)
                generated_code = extract_code(repair_response['text'])
                attempts = 2

        test_passed = executor.run_tests(generated_code, problem['tests'])
        cond_passed += int(test_passed)
        results_table.append({
            'condition': condition,
            'problem_id': problem['id'],
            'category': problem['category'],
            'passed': test_passed,
            'attempts': attempts,
            'prompt_tokens': response['prompt_tokens'],
            'completion_tokens': response['completion_tokens'],
        })
        symbol = 'PASS' if test_passed else 'FAIL'
        print(f"    {symbol:<4} {problem['category']:<18} {problem['id']:<8} attempts={attempts}")

    pass_pct = 100 * cond_passed / len(DEMO_PROBLEMS)
    label = 'FinalPass' if condition == 'C5_iterative_repair' else 'Pass@1'
    print(f'    {label} = {pass_pct:.0f}% ({cond_passed}/{len(DEMO_PROBLEMS)})\n')


---
## 6. Results Visualisation

Charts showing the mini-experiment results. These same charts would be generated
from your real 5-model × 5-condition results using the same code.

In [ ]:
# Compute pass rates per condition
cond_pass = {}
for row in results_table:
    c = row['condition']
    cond_pass.setdefault(c, []).append(row['passed'])

pass_rates = {c: 100 * sum(v) / len(v) for c, v in cond_pass.items()}
short_labels = ['C1\nZero-Shot', 'C2\nFew-Shot', 'C3\nCoT', 'C4\nRAG', 'C5\nRepair']
rates = [pass_rates.get(c, 0) for c in CONDITIONS_DEMO]

# Delta (gain over C1)
deltas = [r - rates[0] for r in rates]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: Absolute Pass@1
colors = ['#3B82F6','#EAB308','#F97316','#8B5CF6','#EF4444']
axes[0].bar(short_labels, rates, color=colors, width=0.55, edgecolor='white')
axes[0].set_ylabel('Pass@1 (%)', fontsize=12)
axes[0].set_title('Pass@1 by Augmentation Condition', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 115)
axes[0].axhline(rates[0], color='gray', linestyle='--', alpha=0.5, label='C1 baseline')
axes[0].legend(fontsize=10)
for i, (bar, val) in enumerate(zip(axes[0].patches, rates)):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 f'{val:.0f}%', ha='center', fontsize=11, fontweight='bold')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Chart 2: Gains over baseline
delta_colors = ['#94A3B8' if d <= 0 else '#22C55E' for d in deltas]
axes[1].bar(short_labels, deltas, color=delta_colors, width=0.55, edgecolor='white')
axes[1].set_ylabel('Δ Pass@1 vs. C1 (percentage points)', fontsize=12)
axes[1].set_title('Gain Over Zero-Shot Baseline', fontsize=13, fontweight='bold')
axes[1].axhline(0, color='black', linewidth=0.8)
for i, (bar, val) in enumerate(zip(axes[1].patches, deltas)):
    ypos = val + 0.5 if val >= 0 else val - 2.5
    axes[1].text(bar.get_x()+bar.get_width()/2, ypos,
                 f'{val:+.0f}pp', ha='center', fontsize=11, fontweight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Token efficiency per condition
cond_tokens = {}
cond_correct = {}
for row in results_table:
    c = row['condition']
    cond_tokens.setdefault(c, 0)
    cond_correct.setdefault(c, 0)
    cond_tokens[c] += row['prompt_tokens'] + row['completion_tokens']
    cond_correct[c] += int(row['passed'])

efficiencies = {c: (cond_correct[c] * 1000) / cond_tokens[c] if cond_tokens[c] else 0
                for c in CONDITIONS_DEMO}
eff_vals = [efficiencies.get(c, 0) for c in CONDITIONS_DEMO]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(short_labels, eff_vals, color=colors, width=0.55, edgecolor='white')
ax.set_ylabel('Correct solutions per 1,000 tokens', fontsize=12)
ax.set_title('Token Efficiency by Condition', fontsize=13, fontweight='bold')
for bar, val in zip(ax.patches, eff_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
print('Interpretation: higher = more correct solutions per token spent (cost efficiency)')

In [ ]:
# Repair convergence: show repair attempt breakdown for C5
c5_rows = [r for r in results_table if r['condition'] == 'C5_iterative_repair']
attempt_counts = {1: 0, 2: 0}
for row in c5_rows:
    attempt_counts[row['attempts']] = attempt_counts.get(row['attempts'], 0) + 1

labels = ['Solved on\nattempt 1', 'Needed\nrepair (attempt 2)']
vals = [attempt_counts.get(1, 0), attempt_counts.get(2, 0)]
pie_colors = ['#22C55E', '#F97316']

fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    vals, labels=labels, colors=pie_colors, autopct='%1.0f%%',
    startangle=90, textprops={'fontsize': 12}
)
ax.set_title('C5 Repair Convergence\n(How many problems needed a repair round?)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Solved first try: {vals[0]}/{sum(vals)} | Needed repair: {vals[1]}/{sum(vals)}')

---
## 6.5 Running the Real 3-Model Experiment

The sections above use **one real example per category** and a mock model so the notebook
stays fast and presentation-friendly. For the actual experiment, use the prepared dataset
in this project folder and the three Ollama model names below.

### Real run command
```bash
python experiment_runner_patched.py --biocoder-dir ./biocoder_dataset --num-problems 80 --models ollama:qwen2.5:0.5b ollama:qwen2.5-coder:1.5b ollama:phi3:mini
```

### Demo vs. Real Results - summary

| | Notebook demo | Real experiment |
|---|---|---|
| Dataset | 4 real examples (1 per category) | 80-207 BioCoder problems |
| Models used | Mock model | Your 3 Ollama models |
| Ollama required | No | Yes |
| Purpose | Live demo, pipeline walkthrough | Research-grade ablation results |
| Reportable in paper | No | Yes |


---
## 7. Summary - What You Just Saw

| Section | What was demonstrated |
|---------|----------------------|
| **2. Code Executor** | Subprocess isolation, pass/fail detection, repair-style error capture, timeout |
| **3. Augmentation Pipeline** | All 5 prompt conditions rendered side-by-side for the same problem |
| **4. Model Interface** | How your 3 Ollama model names map into the interface layer |
| **5. Mini Experiment** | The end-to-end loop: prompt ? model ? test ? optional repair |
| **6. Visualisation** | Pass@1, gain-over-baseline, token efficiency, repair convergence |

### Running the Real Experiment
Once Ollama is running and your three models are available locally:
```bash
python experiment_runner_patched.py --models ollama:qwen2.5:0.5b ollama:qwen2.5-coder:1.5b ollama:phi3:mini --biocoder-dir ./biocoder_dataset --num-problems 80
```
